In [1]:
import os
os.chdir("/home/as5606/projects/CLIP-UP")
import sys
sys.path.insert(0, "/home/as5606/projects/CLIP-UP")

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import torch
from PIL import Image
from clip_up.vlms.clip_up_vlms import init_vlm, get_vlm_path_and_config, run_vlm_inference_for_single_item
from clip_up.clip_up_wrapper import ClipUpWrapper, StructureClipSignalExtractor
from clip_up.clip_up_utils import InjectionType
from clip_up.clip_up_general_utils import seed_everything, disable_torch_init

/home/as5606/miniconda3/envs/clip_up/lib/python3.9/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

qwen_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", torch_dtype="auto", device_map="auto")
qwen_tok   = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

SYSTEM_PROMPT = """You are converting chest X-ray visual-question-answering questions into short 
declarative statements, for use as text queries against a biomedical CLIP model.

Convert the question into ONE short, factual, declarative sentence stating the topic 
or finding being asked about, phrased like a radiology caption.

Rules:
1. Output exactly ONE sentence. No preamble, no explanation.
2. Do NOT invent a specific answer value unless already in the question.
3. Remove any "<image>" placeholder text.
4. Write in third person, declarative tone — like a radiology report.
5. Do not use hedging language ("might", "could", "possibly").
6. Preserve clinical/anatomical terminology from the question.
7. Prefer "The chest X-ray shows/demonstrates ..." as the base template.

Examples:
Question: "Where is the tip of the central line located?"
Statement: "The chest X-ray shows the tip of a central line."

Question: "What is the size of the pneumothorax?"
Statement: "The chest X-ray shows a pneumothorax."

Question: "What can you say about the tracheal position?"
Statement: "The chest X-ray shows the position of the trachea."

Question: "What do the arrows indicate?"
Statement: "The chest X-ray contains arrows indicating a specific finding.\""""


def make_statement(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f'Question: "{question}"'}
    ]
    text   = qwen_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_tok([text], return_tensors="pt").to(qwen_model.device)
    out    = qwen_model.generate(**inputs, max_new_tokens=128)
    resp   = qwen_tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return resp.strip().strip('"').strip("'").split("\n")[0].strip()




Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


In [5]:
CHECKPOINT = "/home/as5606/projects/Super_LlaVA_Med/model_checkpoints/med-llava/open_ended_train_results/dataset3_xray_run1/checkpoint_epoch_2.pt"
IMAGE_PATH = "/shared/rc/vlm-idk/LLaVA/data/images/33014502_fig1.jpg"
QUESTION   = "What can you say about the right costophrenic sinus?"
STATEMENT  = make_statement(QUESTION) # Qwen-converted statement
print(STATEMENT)


The chest X-ray shows the right costophrenic sinus.


In [6]:
device = "cuda"
disable_torch_init()
seed_everything(42, with_accelerate=False)

In [7]:
# Load model
model_path, model_config = get_vlm_path_and_config("med-llava")
model, image_processor, tokenizer, conv_mode = init_vlm(
    model_name="med-llava", model_path=model_path, model_config=model_config,
    is_clip_up_emb_lora_injection=False, is_for_open_ended_questions=True, device=device
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of the model checkpoint at /shared/rc/vlm-idk/LLaVA/checkpoints/llava-med-v1.5-mistral-7b were not used when initializing LlavaMistralForCausalLM: {'model.vision_tower.vision_tower.vision_model.encoder.layers.2.self_attn.out_proj.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.6.layer_norm2.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.23.mlp.fc2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.21.self_attn.k_proj.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.18.mlp.fc1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.mlp.fc1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.6.mlp.fc2.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.19.mlp.fc2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.5.mlp.fc1.weight', 'model.vision_tower.v

In [8]:
# Load CLIP-UP wrapper + checkpoint
wrapper = ClipUpWrapper(model=model, model_name="med-llava", model_config=model_config,
                        conv_mode=conv_mode, tokenizer=tokenizer,
                        image_processor=image_processor, device=device,
                        output_dir="/tmp/inference_out", accelerator=None,
                        is_for_open_ended_questions=True,
                        injection_type=InjectionType.CLIP_UP_EMB)
wrapper.load_checkpoint(CHECKPOINT)
model.requires_grad_(False)
model.eval()

Loading checkpoint from /home/as5606/projects/Super_LlaVA_Med/model_checkpoints/med-llava/open_ended_train_results/dataset3_xray_run1/checkpoint_epoch_2.pt


LlavaMistralForCausalLM(
  (model): LlavaMistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((4096,)

In [9]:
# Extract CLIP signal
extractor = StructureClipSignalExtractor(device=device, torch_dtype=torch.float16,
                                          structure_clip_checkpoint_path=model_config.structure_clip_checkpoint_path)

In [10]:
sample = {"statement": STATEMENT, "image_path": IMAGE_PATH, "hint": None}
clip_signal = extractor.extract_clip_signal([STATEMENT], Image.open(IMAGE_PATH)).flatten()
clip_signal = clip_signal.to(torch.bfloat16)

In [14]:
# Run inference
output, _ = run_vlm_inference_for_single_item(
    model=model, model_name="med-llava", conv_mode=conv_mode,
    tokenizer=tokenizer, image_processor=image_processor,
    upd_setting=wrapper.upd_setting, upd_type="open_ended",
    question=QUESTION, hint=None, options=[], image=Image.open(IMAGE_PATH),
    clip_signal=clip_signal
)
print("Answer:", output)


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Answer: The right costophrenic sinus appears to be blunted in the chest X-ray. This means that the angle between the diaphragm and the chest wall is not as sharp as it should be, which could be due to various reasons such as fluid accumulation, inflammation, or other abnormalities.


In [15]:
import json
from tqdm import tqdm

VAL_PATH = "/home/as5606/projects/Super_LlaVA_Med/dataset_3/x_ray/open_ended_train_and_val/val_open_ended_annotations.jsonl"

val_records = [json.loads(l) for l in open(VAL_PATH)]
standard_records = [r for r in val_records if r["type"] == "standard"]
upd_records      = [r for r in val_records if r["type"] == "upd"]

In [16]:
print(f"Standard: {len(standard_records)}, UPD: {len(upd_records)}")


Standard: 385, UPD: 385


In [17]:
def run_on_records(records, label):
    results = []
    for rec in tqdm(records, desc=label):
        statement  = rec["statement"]
        clip_signal = extractor.extract_clip_signal([statement], Image.open(rec["image_path"])).flatten()
        clip_signal = clip_signal.to(torch.bfloat16)

        output, _ = run_vlm_inference_for_single_item(
            model=model, model_name="med-llava", conv_mode=conv_mode,
            tokenizer=tokenizer, image_processor=image_processor,
            upd_setting=wrapper.upd_setting, upd_type="open_ended",
            question=rec["question"], hint=None, options=[],
            image=Image.open(rec["image_path"]),
            clip_signal=clip_signal
        )
        results.append({
            "question":   rec["question"],
            "gt_answer":  rec["answer"],
            "prediction": output,
            "type":       rec["type"],
        })
        print(f"Q: {rec['question']}")
        print(f"GT: {rec['answer']}")
        print(f"Pred: {output}")
        print("---")
    return results

[{'question': 'What is the catheter placed in?',
  'gt_answer': 'The catheter was inadvertently placed via the right carotid artery into the thoracic aorta.',
  'prediction': 'The catheter is placed in the right internal jugular vein.',
  'type': 'standard'},
 {'question': 'What is the issue with the catheter placement?',
  'gt_answer': 'The issue with the catheter placement in this image is that it has been misplaced into the vertebral vein. The catheter passes the transverse processes of the 6th and 7th cervical vertebrae, which is not the intended location for a central venous catheter.',
  'prediction': 'The issue with the catheter placement in the image is that the tip of the catheter is located in the right atrium, which is not the intended position. The catheter should ideally be placed in the superior vena cava (SVC) to ensure proper functioning.',
  'type': 'standard'},
 {'question': 'Where does the lead terminate?',
  'gt_answer': 'The lead of the transvenous pacing wire term

In [18]:
standard_results = run_on_records(standard_records, "Standard")


Standard:   0%|                                                                                    | 0/385 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Standard:   0%|▏                                                                           | 1/385 [00:00<04:03,  1.58it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q: What is the catheter placed in?
GT: The catheter was inadvertently placed via the right carotid artery into the thoracic aorta.
Pred: The catheter is placed in the right internal jugular vein.
---


Standard:   1%|▍                                                                           | 2/385 [00:02<09:03,  1.42s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q: What is the issue with the catheter placement?
GT: The issue with the catheter placement in this image is that it has been misplaced into the vertebral vein. The catheter passes the transverse processes of the 6th and 7th cervical vertebrae, which is not the intended location for a central venous catheter.
Pred: The issue with the catheter placement in the image is that the tip of the catheter is located in the right atrium, which is not the intended position. The catheter should ideally be placed in the superior vena cava (SVC) to ensure proper functioning.
---


Standard:   1%|▌                                                                           | 3/385 [00:03<06:27,  1.01s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

Q: What about the effusion?
GT: There is a small effusion present in the right costophrenic angle. The costophrenic angle is the area where the diaphragm meets the rib cage, and it is a common location for fluid to accumulate when there is an effusion.
Pred: The chest X-ray shows a resolution of the effusion, which means that the fluid accumulation in the pleural space has been reduced or eliminated.
---


Standard: 100%|█████████████████████████████████████████████████████████████████████████▊| 384/385 [07:19<00:01,  1.11s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q: What happened to the trachea?
GT: In the image, the trachea is pushed to the left side due to the right tension pneumothorax and the resulting mediastinal shift.
Pred: I cannot answer.
---


Standard: 100%|██████████████████████████████████████████████████████████████████████████| 385/385 [07:20<00:00,  1.14s/it]

Q: What is the position of the LV lead?
GT: The left ventricle (LV) lead is implanted in the lateral coronary vein (LCV).
Pred: I cannot answer.
---


In [20]:
len(standard_results)

385

In [26]:
c= 0

for g in standard_results:
    if g['prediction'] == "I cannot answer.":
        c += 1
    
print(c)

66


In [27]:
66/len(standard_results)

0.17142857142857143

In [28]:
upd_results  = run_on_records(upd_records, "UPD")


UPD:   0%|                                                                                         | 0/385 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
UPD:   0%|▏                                                                                | 1/385 [00:00<03:40,  1.74it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q: What is the condition of the pneumomediastinum and subcutaneous emphysema in the image?
GT: I cannot answer.
Pred: I cannot answer.
---


UPD:   1%|▍                                                                                | 2/385 [00:00<02:36,  2.45it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q: Is the pleural effusion on one side or both sides?
GT: I cannot answer.
Pred: I cannot answer.
---


UPD:   1%|▋                                                                                | 3/385 [00:01<02:11,  2.91it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

Q: Are the infiltrates present in one lung or both?
GT: I cannot answer.
Pred: The infiltrates are present in both lungs, as they are described as bilateral infiltrates.
---


UPD: 100%|██████████████████████████████████████████████████████████████████████████████▊| 384/385 [02:35<00:00,  2.32it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q: What is the RALE score in the image?
GT: I cannot answer.
Pred: I cannot answer.
---


UPD: 100%|███████████████████████████████████████████████████████████████████████████████| 385/385 [02:35<00:00,  2.47it/s]

Q: What is the left hilar density?
GT: I cannot answer.
Pred: I cannot answer.
---


In [29]:
c= 0

for g in upd_results:
    if g['prediction'] != "I cannot answer.":
        c += 1
    
print(c)


49
